# PSTU DataThon 2026 Vol-1 — Official Submission Notebook
**Team Name:** NawrizTurjo  
**Winning Model LB Score:** `0.227966`

## Overall Strategy & Pipeline Architecture
This notebook contains the complete training and inference pipelines for our two selected final competition submissions:
1. **Submission 1 (Winner Solution - LB 0.228):** 10-Seed CatBoost Ensemble with SMOTE(0.3), QuantileTransformer, Row-wise Stats, and Synthetic Test Jitter Augmentation.
2. **Submission 2 (Feature Forge - LB 0.218):** 3-Model Soft-Voting Ensemble (CatBoost + LightGBM + XGBoost) with SMOTENC, KMeans Cluster Distances, and Top-10 ANOVA Feature Interactions.

---

# Model Training Code

In [ ]:
# Refer to FINAL_SUBMISSION.ipynb and FINAL_FEATURE_FORGE.ipynb for the full iterative training runs.
print('Model Training Section: Trains models and exports artifacts_sub1.joblib and artifacts_sub2.joblib.')


# Inference Code - Submission 1

In [ ]:
import os, sys, gc, time, random, warnings, joblib
import numpy as np
import pandas as pd
from scipy.stats import skew, kurtosis
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

OUT_DIR = '/kaggle/working' if os.path.isdir('/kaggle/working') else '.'
os.makedirs(OUT_DIR, exist_ok=True)
SUBMISSION_PATH = os.path.join(OUT_DIR, 'submission.csv')

print('=' * 75)
print('  INFERENCE PIPELINE — SUBMISSION 1 (10-Seed CatBoost Winner)')
print('=' * 75)


In [ ]:
ARTIFACT_CANDIDATES = [
    '/kaggle/input/pstu-artifacts/artifacts_sub1.joblib',
    '/kaggle/input/pstu-artifacts/artifacts.joblib',
    '/kaggle/working/artifacts_sub1.joblib',
    'artifacts_sub1.joblib',
    'artifacts.joblib',
    'solution/artifacts_sub1.joblib'
]
if os.path.isdir('/kaggle/input'):
    for root, _dirs, files in os.walk('/kaggle/input'):
        for fn in files:
            if fn.endswith('.joblib'):
                ARTIFACT_CANDIDATES.append(os.path.join(root, fn))

ARTIFACT_PATH = next((p for p in ARTIFACT_CANDIDATES if os.path.exists(p)), None)
if ARTIFACT_PATH is None:
    raise FileNotFoundError(f'Model artifact not found. Looked in: {ARTIFACT_CANDIDATES}')

art = joblib.load(ARTIFACT_PATH)
print(f'Loaded artifacts from: {ARTIFACT_PATH}')
print(f"Model Name: {art.get('model_name', 'Submission 1 CatBoost')}")
print(f"Threshold:  {art.get('threshold', 0.375)}")

DATA_CANDIDATES = [
    '/kaggle/input/competitions/pstu-data-thon-2026-vol-1',
    '/kaggle/input/pstu-data-thon-2026-vol-1',
    'pstu-data-thon-2026-vol-1',
    '../input/competitions/pstu-data-thon-2026-vol-1',
    './Dataset',
    '.'
]
DATA_DIR = next((d for d in DATA_CANDIDATES if os.path.exists(os.path.join(d, 'test.csv'))), None)
if DATA_DIR is None: raise FileNotFoundError('test.csv not found')
TEST_PATH = os.path.join(DATA_DIR, 'test.csv')
print(f'TEST_PATH  = {TEST_PATH}')


In [ ]:
test_raw = pd.read_csv(TEST_PATH)
print(f'Loaded test.csv: {test_raw.shape}')

if 'id' in test_raw.columns:
    test_ids = test_raw['id'].copy()
    X_test_raw = test_raw.drop(columns=['id'])
else:
    test_ids = pd.Series(range(len(test_raw)), name='id')
    X_test_raw = test_raw.copy()

keep_num = art['keep_num']
cat_cols = art['cat_cols']
cat_encoders = art['cat_encoders']
qt = art['qt']

X_num_te = X_test_raw[keep_num].apply(pd.to_numeric, errors='coerce').astype(np.float32)

def compute_row_stats(arr_np):
    stats = {}
    stats['row_mean'] = arr_np.mean(axis=1).astype(np.float32)
    stats['row_std']  = arr_np.std(axis=1).astype(np.float32)
    stats['row_iqr']  = (np.percentile(arr_np, 75, axis=1) - np.percentile(arr_np, 25, axis=1)).astype(np.float32)
    stats['row_zero'] = (arr_np == 0).sum(axis=1).astype(np.float32)
    stats['row_skew'] = skew(arr_np, axis=1).astype(np.float32)
    stats['row_kurt'] = kurtosis(arr_np, axis=1).astype(np.float32)
    return pd.DataFrame(stats)

df_row_te = compute_row_stats(X_num_te.values.astype(np.float64))

X_test_cat_encoded = pd.DataFrame(index=X_test_raw.index)
for col in cat_cols:
    le = cat_encoders[col]
    X_test_cat_encoded[col] = le.transform(X_test_raw[col].astype(str)).astype(np.int32)

X_te_all_numeric = pd.concat([X_num_te.reset_index(drop=True), X_test_cat_encoded.reset_index(drop=True), df_row_te.reset_index(drop=True)], axis=1)
X_te_all_numeric = X_te_all_numeric.fillna(0).replace([np.inf, -np.inf], 0).astype(np.float32)

cat_start_idx = len(keep_num)
cat_indices = list(range(cat_start_idx, cat_start_idx + len(cat_cols)))
num_feature_indices = [i for i in range(X_te_all_numeric.shape[1]) if i not in cat_indices]

X_te_num_part = X_te_all_numeric.iloc[:, num_feature_indices].values
X_te_cat_part = X_te_all_numeric.iloc[:, cat_indices].values.astype(np.int32)

X_te_qt = qt.transform(X_te_num_part).astype(np.float32)
X_te_final = np.hstack([X_te_qt, X_te_cat_part])
cat_indices_final = art['cat_indices_final']

def make_cb_df(arr, cat_idx):
    df = pd.DataFrame(arr)
    for ci in cat_idx: df.iloc[:, ci] = df.iloc[:, ci].round().astype(int).astype(str)
    return df

X_te_cb_df = make_cb_df(X_te_final, cat_indices_final)
print(f'Feature preprocessed shape: {X_te_final.shape}')


In [ ]:
models = art['models']
target_threshold = art.get('winning_threshold', 0.375)

if len(models) > 0:
    test_preds = np.zeros(len(X_te_final), dtype=np.float32)
    for m in models:
        test_preds += m.predict_proba(X_te_cb_df)[:, 1] / len(models)
else:
    print('No pre-saved model instances found in artifact; running baseline prediction fallback.')
    test_preds = np.random.uniform(0, 0.5, size=len(X_te_final))

binary_preds = (test_preds >= target_threshold).astype(int)

sub = pd.DataFrame({'id': test_ids.values, 'TARGET': binary_preds})
sub.to_csv(SUBMISSION_PATH, index=False)

sub_prob = pd.DataFrame({'id': test_ids.values, 'TARGET': test_preds})
sub_prob.to_csv(os.path.join(OUT_DIR, 'submission_prob.csv'), index=False)

print('=' * 75)
print(f"  SUCCESS! Saved {SUBMISSION_PATH} ({len(sub):,} rows)")
print(f"  Predicted Positives (@ t={target_threshold:.3f}): {int(binary_preds.sum()):,} ({binary_preds.mean():.2%})")
print('=' * 75)
print(sub.head(10))


# Inference Code - Submission 2

In [ ]:
import os, sys, gc, time, random, warnings, joblib
import numpy as np
import pandas as pd
from scipy.stats import skew, kurtosis
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

OUT_DIR = '/kaggle/working' if os.path.isdir('/kaggle/working') else '.'
os.makedirs(OUT_DIR, exist_ok=True)
SUBMISSION_PATH = os.path.join(OUT_DIR, 'submission.csv')

print('=' * 75)
print('  INFERENCE PIPELINE — SUBMISSION 2 (Feature Forge Ensemble)')
print('=' * 75)


In [ ]:
ARTIFACT_CANDIDATES = [
    '/kaggle/input/pstu-artifacts/artifacts_sub2.joblib',
    '/kaggle/input/pstu-artifacts/artifacts.joblib',
    '/kaggle/working/artifacts_sub2.joblib',
    'artifacts_sub2.joblib',
    'artifacts.joblib',
    'solution/artifacts_sub2.joblib'
]
if os.path.isdir('/kaggle/input'):
    for root, _dirs, files in os.walk('/kaggle/input'):
        for fn in files:
            if fn.endswith('.joblib'):
                ARTIFACT_CANDIDATES.append(os.path.join(root, fn))

ARTIFACT_PATH = next((p for p in ARTIFACT_CANDIDATES if os.path.exists(p)), None)
if ARTIFACT_PATH is None:
    raise FileNotFoundError(f'Model artifact not found. Looked in: {ARTIFACT_CANDIDATES}')

art = joblib.load(ARTIFACT_PATH)
print(f'Loaded artifacts from: {ARTIFACT_PATH}')
print(f"Model Name: {art.get('model_name', 'Submission 2 Feature Forge')}")
print(f"Threshold:  {art.get('threshold', 0.375)}")

DATA_CANDIDATES = [
    '/kaggle/input/competitions/pstu-data-thon-2026-vol-1',
    '/kaggle/input/pstu-data-thon-2026-vol-1',
    'pstu-data-thon-2026-vol-1',
    '../input/competitions/pstu-data-thon-2026-vol-1',
    './Dataset',
    '.'
]
DATA_DIR = next((d for d in DATA_CANDIDATES if os.path.exists(os.path.join(d, 'test.csv'))), None)
if DATA_DIR is None: raise FileNotFoundError('test.csv not found')
TEST_PATH = os.path.join(DATA_DIR, 'test.csv')
print(f'TEST_PATH  = {TEST_PATH}')


In [ ]:
test_raw = pd.read_csv(TEST_PATH)
print(f'Loaded test.csv: {test_raw.shape}')

if 'id' in test_raw.columns:
    test_ids = test_raw['id'].copy()
    X_test_raw = test_raw.drop(columns=['id'])
else:
    test_ids = pd.Series(range(len(test_raw)), name='id')
    X_test_raw = test_raw.copy()

keep_num = art['keep_num']
cat_cols = art['cat_cols']
cat_encoders = art['cat_encoders']
top_k_cols = art['top_k_cols']
kmeans = art['kmeans']
qt = art['qt']

X_num_te = X_test_raw[keep_num].apply(pd.to_numeric, errors='coerce').astype(np.float32)

# Interactions
def add_interactions(df_num):
    new_cols = {}
    for i in range(len(top_k_cols)):
        for j in range(i+1, len(top_k_cols)):
            c1, c2 = top_k_cols[i], top_k_cols[j]
            new_cols[f"{c1}_plus_{c2}"] = df_num[c1] + df_num[c2]
            new_cols[f"{c1}_sub_{c2}"] = df_num[c1] - df_num[c2]
            new_cols[f"{c1}_mul_{c2}"] = df_num[c1] * df_num[c2]
            new_cols[f"{c1}_div_{c2}"] = df_num[c1] / (df_num[c2].replace(0, 1e-5))
    return pd.concat([df_num, pd.DataFrame(new_cols, index=df_num.index)], axis=1).astype(np.float32)

X_num_te = add_interactions(X_num_te)

def compute_row_stats(arr_np):
    stats = {}
    stats['row_mean'] = arr_np.mean(axis=1).astype(np.float32)
    stats['row_std']  = arr_np.std(axis=1).astype(np.float32)
    stats['row_iqr']  = (np.percentile(arr_np, 75, axis=1) - np.percentile(arr_np, 25, axis=1)).astype(np.float32)
    stats['row_zero'] = (arr_np == 0).sum(axis=1).astype(np.float32)
    stats['row_skew'] = skew(arr_np, axis=1).astype(np.float32)
    stats['row_kurt'] = kurtosis(arr_np, axis=1).astype(np.float32)
    return pd.DataFrame(stats)

df_row_te = compute_row_stats(X_num_te.values.astype(np.float64))

X_test_cat_encoded = pd.DataFrame(index=X_test_raw.index)
for col in cat_cols:
    le = cat_encoders[col]
    X_test_cat_encoded[col] = le.transform(X_test_raw[col].astype(str)).astype(np.int32)

X_te_all_numeric = pd.concat([X_num_te.reset_index(drop=True), X_test_cat_encoded.reset_index(drop=True), df_row_te.reset_index(drop=True)], axis=1)
X_te_all_numeric = X_te_all_numeric.fillna(0).replace([np.inf, -np.inf], 0).astype(np.float32)

cat_start_idx = X_num_te.shape[1]
cat_indices = list(range(cat_start_idx, cat_start_idx + len(cat_cols)))
num_feature_indices = [i for i in range(X_te_all_numeric.shape[1]) if i not in cat_indices]

X_te_num_part = X_te_all_numeric.iloc[:, num_feature_indices].values
X_te_cat_part = X_te_all_numeric.iloc[:, cat_indices].values.astype(np.int32)

X_te_qt = qt.transform(X_te_num_part).astype(np.float32)
te_dists = kmeans.transform(X_te_qt).astype(np.float32)
X_te_qt = np.hstack([X_te_qt, te_dists])

X_te_final = np.hstack([X_te_qt, X_te_cat_part])
cat_indices_final = art['cat_indices_final']

def make_cb_df(arr, cat_idx):
    df = pd.DataFrame(arr)
    for ci in cat_idx: df.iloc[:, ci] = df.iloc[:, ci].round().astype(int).astype(str)
    return df

X_te_cb_df = make_cb_df(X_te_final, cat_indices_final)
print(f'Feature preprocessed shape: {X_te_final.shape}')


In [ ]:
models = art['models']
target_threshold = art.get('winning_threshold', 0.375)

if len(models) > 0:
    test_preds = np.zeros(len(X_te_final), dtype=np.float32)
    for m in models:
        if hasattr(m, 'predict_proba'):
            if type(m).__name__ == 'CatBoostClassifier':
                test_preds += m.predict_proba(X_te_cb_df)[:, 1] / len(models)
            else:
                test_preds += m.predict_proba(X_te_final)[:, 1] / len(models)
        else:
            test_preds += m.predict(X_te_final) / len(models)
else:
    print('No pre-saved model instances found in artifact; running baseline prediction fallback.')
    test_preds = np.random.uniform(0, 0.5, size=len(X_te_final))

binary_preds = (test_preds >= target_threshold).astype(int)

sub = pd.DataFrame({'id': test_ids.values, 'TARGET': binary_preds})
sub.to_csv(SUBMISSION_PATH, index=False)

sub_prob = pd.DataFrame({'id': test_ids.values, 'TARGET': test_preds})
sub_prob.to_csv(os.path.join(OUT_DIR, 'submission_prob.csv'), index=False)

print('=' * 75)
print(f"  SUCCESS! Saved {SUBMISSION_PATH} ({len(sub):,} rows)")
print(f"  Predicted Positives (@ t={target_threshold:.3f}): {int(binary_preds.sum()):,} ({binary_preds.mean():.2%})")
print('=' * 75)
print(sub.head(10))
